# 01 — Preparación de datos

## Actividad física de ocio y síntomas depresivos

Este notebook construye el dataset analítico del proyecto a partir de tres archivos originales de NHANES:

- `DEMO_L.xpt`: edad e identificador.
- `PAQ_L.xpt`: actividad física moderada y vigorosa de ocio.
- `DPQ_L.xpt`: nueve ítems del PHQ-9.

**Entrada:** `../data/raw/`  
**Salida:** `../data/processed/nhanes_actividad_depresion_final.csv`

## 1. Librerías

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda valor: f"{valor:.2f}")

## 2. Rutas del proyecto

In [ ]:
carpeta_raw = Path("../data/raw")
carpeta_processed = Path("../data/processed")

carpeta_processed.mkdir(parents=True, exist_ok=True)

ruta_demo = carpeta_raw / "DEMO_L.xpt"
ruta_paq = carpeta_raw / "PAQ_L.xpt"
ruta_dpq = carpeta_raw / "DPQ_L.xpt"

ruta_salida = carpeta_processed / "nhanes_actividad_depresion_final.csv"

for ruta in [ruta_demo, ruta_paq, ruta_dpq]:
    if not ruta.exists():
        raise FileNotFoundError(f"No se ha encontrado el archivo: {ruta}")

print("Archivos originales encontrados.")

## 3. Carga de los archivos originales

Se utiliza `encoding="utf-8"` para que las unidades de frecuencia de actividad se lean como texto.

In [ ]:
demo_raw = pd.read_sas(
    ruta_demo,
    format="xport",
    encoding="utf-8"
)

paq_raw = pd.read_sas(
    ruta_paq,
    format="xport",
    encoding="utf-8"
)

dpq_raw = pd.read_sas(
    ruta_dpq,
    format="xport",
    encoding="utf-8"
)

print("DEMO_L:", demo_raw.shape)
print("PAQ_L:", paq_raw.shape)
print("DPQ_L:", dpq_raw.shape)

## 4. Corrección de falsos ceros

Algunos archivos XPT pueden contener números extremadamente pequeños que representan ceros. La función los sustituye por `0.0`.

In [ ]:
def corregir_ceros_xpt(df, umbral=1e-70):
    df = df.copy()

    columnas_numericas = df.select_dtypes(
        include=np.number
    ).columns

    df[columnas_numericas] = df[columnas_numericas].mask(
        df[columnas_numericas].abs() < umbral,
        0.0
    )

    return df

In [ ]:
demo_raw = corregir_ceros_xpt(demo_raw)
paq_raw = corregir_ceros_xpt(paq_raw)
dpq_raw = corregir_ceros_xpt(dpq_raw)

## 5. Selección de variables

Solo se conservan las variables necesarias para responder a la pregunta del proyecto.

In [ ]:
demo = demo_raw[
    ["SEQN", "RIDAGEYR"]
].copy()

paq = paq_raw[
    [
        "SEQN",
        "PAD790Q", "PAD790U", "PAD800",
        "PAD810Q", "PAD810U", "PAD820"
    ]
].copy()

dpq = dpq_raw[
    [
        "SEQN",
        "DPQ010", "DPQ020", "DPQ030",
        "DPQ040", "DPQ050", "DPQ060",
        "DPQ070", "DPQ080", "DPQ090"
    ]
].copy()

## 6. Comprobación de identificadores

Cada participante debe aparecer una sola vez en cada tabla.

In [ ]:
print("Duplicados DEMO:", demo["SEQN"].duplicated().sum())
print("Duplicados PAQ:", paq["SEQN"].duplicated().sum())
print("Duplicados DPQ:", dpq["SEQN"].duplicated().sum())

assert demo["SEQN"].is_unique
assert paq["SEQN"].is_unique
assert dpq["SEQN"].is_unique

## 7. Limpieza de códigos especiales

Los códigos de respuesta desconocida o rechazada se convierten en valores nulos.

In [ ]:
columnas_paq_numericas = [
    "PAD790Q",
    "PAD800",
    "PAD810Q",
    "PAD820"
]

paq[columnas_paq_numericas] = (
    paq[columnas_paq_numericas]
    .replace([7777, 9999], np.nan)
)

for columna in ["PAD790U", "PAD810U"]:
    paq[columna] = (
        paq[columna]
        .replace(r"^\s*$", np.nan, regex=True)
    )

columnas_phq_originales = [
    "DPQ010", "DPQ020", "DPQ030",
    "DPQ040", "DPQ050", "DPQ060",
    "DPQ070", "DPQ080", "DPQ090"
]

dpq[columnas_phq_originales] = (
    dpq[columnas_phq_originales]
    .replace([7, 9], np.nan)
)

## 8. Selección de adultos e integración

Se incluyen participantes de **20 años o más**. La edad se utiliza únicamente para aplicar este criterio y no se conserva en el dataset final.

In [ ]:
demo_adultos = demo.loc[
    demo["RIDAGEYR"] >= 20
].copy()

df = (
    demo_adultos
    .merge(
        paq,
        on="SEQN",
        how="left",
        validate="one_to_one"
    )
    .merge(
        dpq,
        on="SEQN",
        how="left",
        validate="one_to_one"
    )
)

print("Adultos de 20 años o más:", len(demo_adultos))
print("Dimensiones tras la integración:", df.shape)

## 9. Construcción del PHQ-9

Cada ítem toma valores de 0 a 3. La puntuación total solo se calcula cuando están disponibles los nueve ítems.

In [ ]:
renombrado_phq = {
    "DPQ010": "phq_interes_placer",
    "DPQ020": "phq_animo_deprimido",
    "DPQ030": "phq_problemas_sueno",
    "DPQ040": "phq_fatiga_energia",
    "DPQ050": "phq_apetito",
    "DPQ060": "phq_autoestima_fracaso",
    "DPQ070": "phq_concentracion",
    "DPQ080": "phq_alteracion_psicomotora",
    "DPQ090": "phq_pensamientos_muerte_autolesion"
}

df = df.rename(columns=renombrado_phq)

columnas_phq = list(
    renombrado_phq.values()
)

df["phq9"] = df[columnas_phq].sum(
    axis=1,
    min_count=9
)

In [ ]:
df["nivel_sintomas_depresivos"] = pd.cut(
    df["phq9"],
    bins=[-1, 4, 9, 14, 19, 27],
    labels=[
        "Mínimos",
        "Leves",
        "Moderados",
        "Moderadamente graves",
        "Graves"
    ]
)

df["sintomas_depresivos_relevantes"] = pd.NA

mascara_phq_completo = df["phq9"].notna()

df.loc[
    mascara_phq_completo,
    "sintomas_depresivos_relevantes"
] = np.where(
    df.loc[
        mascara_phq_completo,
        "phq9"
    ] >= 10,
    "Sí",
    "No"
)

df["numero_sintomas_presentes"] = (
    df[columnas_phq]
    .gt(0)
    .sum(axis=1)
    .where(df["phq9"].notna())
)

df["numero_sintomas_frecuentes"] = (
    df[columnas_phq]
    .ge(2)
    .sum(axis=1)
    .where(df["phq9"].notna())
)

## 10. Conversión de la actividad a frecuencia semanal

Las respuestas pueden estar expresadas por día, semana, mes o año. Se convierten a sesiones por semana.

In [ ]:
def convertir_a_semana(frecuencia, unidad):
    if pd.isna(frecuencia):
        return np.nan

    if frecuencia == 0:
        return 0.0

    if pd.isna(unidad):
        return np.nan

    if isinstance(unidad, bytes):
        unidad = unidad.decode("utf-8")

    unidad = str(unidad).strip().upper()

    factores = {
        "D": 7,
        "W": 1,
        "M": 12 / 52,
        "Y": 1 / 52
    }

    factor = factores.get(unidad)

    if factor is None:
        return np.nan

    return frecuencia * factor

In [ ]:
df["sesiones_moderadas_semana"] = df.apply(
    lambda fila: convertir_a_semana(
        fila["PAD790Q"],
        fila["PAD790U"]
    ),
    axis=1
)

df["sesiones_vigorosas_semana"] = df.apply(
    lambda fila: convertir_a_semana(
        fila["PAD810Q"],
        fila["PAD810U"]
    ),
    axis=1
)

## 11. Duración y minutos semanales

Cuando la frecuencia declarada es cero, la duración y los minutos semanales se consideran cero. Si hay actividad pero falta la duración, el componente no es interpretable.

In [ ]:
def preparar_duracion(sesiones_semana, duracion_original):
    if pd.isna(sesiones_semana):
        return np.nan

    if sesiones_semana == 0:
        return 0.0

    if pd.isna(duracion_original):
        return np.nan

    return duracion_original

In [ ]:
df["duracion_moderada_sesion_min"] = df.apply(
    lambda fila: preparar_duracion(
        fila["sesiones_moderadas_semana"],
        fila["PAD800"]
    ),
    axis=1
)

df["duracion_vigorosa_sesion_min"] = df.apply(
    lambda fila: preparar_duracion(
        fila["sesiones_vigorosas_semana"],
        fila["PAD820"]
    ),
    axis=1
)

df["actividad_moderada_ocio_min_semana"] = (
    df["sesiones_moderadas_semana"]
    * df["duracion_moderada_sesion_min"]
)

df["actividad_vigorosa_ocio_min_semana"] = (
    df["sesiones_vigorosas_semana"]
    * df["duracion_vigorosa_sesion_min"]
)

## 12. Selección de la muestra analítica

La muestra final requiere:

1. actividad moderada interpretable;
2. actividad vigorosa interpretable;
3. los nueve ítems del PHQ-9 completos.

In [ ]:
mascara_actividad_valida = (
    df["actividad_moderada_ocio_min_semana"].notna()
    & df["actividad_vigorosa_ocio_min_semana"].notna()
)

df_actividad_valida = df.loc[
    mascara_actividad_valida
].copy()

df_final = df_actividad_valida.loc[
    df_actividad_valida["phq9"].notna()
].copy()

flujo_muestra = pd.DataFrame(
    {
        "etapa": [
            "Adultos de 20 años o más",
            "Actividad moderada y vigorosa interpretables",
            "Actividad interpretable y PHQ-9 completo"
        ],
        "participantes": [
            len(df),
            len(df_actividad_valida),
            len(df_final)
        ]
    }
)

flujo_muestra

## 13. Variables derivadas de actividad

- **Actividad total:** minutos reales de actividad moderada y vigorosa.
- **Actividad equivalente:** minutos moderados + dos veces los minutos vigorosos. Se utiliza para clasificar el nivel de actividad; no representa tiempo real transcurrido.

In [ ]:
df_final["sesiones_totales_semana"] = (
    df_final["sesiones_moderadas_semana"]
    + df_final["sesiones_vigorosas_semana"]
)

df_final["actividad_total_ocio_min_semana"] = (
    df_final["actividad_moderada_ocio_min_semana"]
    + df_final["actividad_vigorosa_ocio_min_semana"]
)

df_final["actividad_equivalente_ocio_min_semana"] = (
    df_final["actividad_moderada_ocio_min_semana"]
    + 2 * df_final["actividad_vigorosa_ocio_min_semana"]
)

df_final["duracion_media_sesion_min"] = np.where(
    df_final["sesiones_totales_semana"] > 0,
    (
        df_final["actividad_total_ocio_min_semana"]
        / df_final["sesiones_totales_semana"]
    ),
    0.0
)

## 14. Redondeo y clasificación de la actividad

In [ ]:
columnas_actividad = [
    "sesiones_moderadas_semana",
    "duracion_moderada_sesion_min",
    "actividad_moderada_ocio_min_semana",
    "sesiones_vigorosas_semana",
    "duracion_vigorosa_sesion_min",
    "actividad_vigorosa_ocio_min_semana",
    "sesiones_totales_semana",
    "duracion_media_sesion_min",
    "actividad_total_ocio_min_semana",
    "actividad_equivalente_ocio_min_semana"
]

df_final[columnas_actividad] = (
    df_final[columnas_actividad]
    .round(2)
)

equivalente = df_final[
    "actividad_equivalente_ocio_min_semana"
]

condiciones_actividad = [
    equivalente == 0,
    (equivalente > 0) & (equivalente < 150),
    (equivalente >= 150) & (equivalente < 300),
    equivalente >= 300
]

df_final["nivel_actividad"] = np.select(
    condiciones_actividad,
    [
        "Inactivo",
        "Insuficiente",
        "Recomendado",
        "Alto"
    ],
    default="Sin clasificar"
)

In [ ]:
moderada_positiva = (
    df_final["actividad_moderada_ocio_min_semana"] > 0
)

vigorosa_positiva = (
    df_final["actividad_vigorosa_ocio_min_semana"] > 0
)

df_final["perfil_actividad"] = np.select(
    [
        ~moderada_positiva & ~vigorosa_positiva,
        moderada_positiva & ~vigorosa_positiva,
        ~moderada_positiva & vigorosa_positiva,
        moderada_positiva & vigorosa_positiva
    ],
    [
        "Inactivo",
        "Solo moderada",
        "Solo vigorosa",
        "Moderada + vigorosa"
    ],
    default="Sin clasificar"
)

df_final["alcanza_umbral_actividad"] = np.where(
    equivalente >= 150,
    "Sí",
    "No"
)

limite_p99 = (
    df_final["actividad_total_ocio_min_semana"]
    .quantile(0.99)
)

df_final["flag_actividad_extrema"] = np.where(
    df_final["actividad_total_ocio_min_semana"] > limite_p99,
    "Sí",
    "No"
)

## 15. Selección de columnas y tipos

In [ ]:
columnas_finales = [
    "SEQN",
    "sesiones_moderadas_semana",
    "duracion_moderada_sesion_min",
    "actividad_moderada_ocio_min_semana",
    "sesiones_vigorosas_semana",
    "duracion_vigorosa_sesion_min",
    "actividad_vigorosa_ocio_min_semana",
    "sesiones_totales_semana",
    "duracion_media_sesion_min",
    "actividad_total_ocio_min_semana",
    "actividad_equivalente_ocio_min_semana",
    "nivel_actividad",
    "perfil_actividad",
    "alcanza_umbral_actividad",
    "flag_actividad_extrema",
    *columnas_phq,
    "phq9",
    "nivel_sintomas_depresivos",
    "sintomas_depresivos_relevantes",
    "numero_sintomas_presentes",
    "numero_sintomas_frecuentes"
]

df_final = (
    df_final[columnas_finales]
    .rename(columns={"SEQN": "id"})
    .copy()
)

columnas_enteras = [
    "id",
    *columnas_phq,
    "phq9",
    "numero_sintomas_presentes",
    "numero_sintomas_frecuentes"
]

df_final[columnas_enteras] = (
    df_final[columnas_enteras]
    .astype("int64")
)

## 16. Validación final

Las comprobaciones detienen la ejecución si se detecta una incoherencia.

In [ ]:
assert df_final["id"].is_unique
assert not df_final.isna().any().any()

assert df_final["phq9"].between(0, 27).all()

assert (
    df_final[columnas_phq]
    .apply(lambda columna: columna.between(0, 3).all())
    .all()
)

assert (
    df_final[columnas_phq]
    .sum(axis=1)
    .eq(df_final["phq9"])
    .all()
)

assert (
    df_final[columnas_phq]
    .gt(0)
    .sum(axis=1)
    .eq(df_final["numero_sintomas_presentes"])
    .all()
)

assert (
    df_final[columnas_phq]
    .ge(2)
    .sum(axis=1)
    .eq(df_final["numero_sintomas_frecuentes"])
    .all()
)

assert np.allclose(
    df_final["actividad_total_ocio_min_semana"],
    (
        df_final["actividad_moderada_ocio_min_semana"]
        + df_final["actividad_vigorosa_ocio_min_semana"]
    ),
    atol=0.01
)

assert np.allclose(
    df_final["actividad_equivalente_ocio_min_semana"],
    (
        df_final["actividad_moderada_ocio_min_semana"]
        + 2 * df_final["actividad_vigorosa_ocio_min_semana"]
    ),
    atol=0.01
)

assert np.allclose(
    df_final["sesiones_totales_semana"],
    (
        df_final["sesiones_moderadas_semana"]
        + df_final["sesiones_vigorosas_semana"]
    ),
    atol=0.01
)

assert "Sin clasificar" not in df_final["nivel_actividad"].values
assert "Sin clasificar" not in df_final["perfil_actividad"].values

print("Validación completada correctamente.")

## 17. Resumen y exportación

In [ ]:
print("Filas:", df_final.shape[0])
print("Columnas:", df_final.shape[1])
print("Valores nulos:", df_final.isna().sum().sum())
print("IDs duplicados:", df_final["id"].duplicated().sum())
print("P99 de actividad total:", round(limite_p99, 2))

print("\nNivel de actividad:")
display(df_final["nivel_actividad"].value_counts())

print("\nNivel de síntomas depresivos:")
display(df_final["nivel_sintomas_depresivos"].value_counts())

display(df_final.head())

In [ ]:
df_final.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset exportado en:", ruta_salida)

## Resultado

El archivo generado contiene una fila por participante y las variables definitivas utilizadas en los notebooks de análisis.